# 🧠 Multimodal Deep Learning Tutorial: Vision + Text Fusion Architecture

Welcome to this comprehensive tutorial on **Multimodal Deep Learning** using **PyTorch**, **NumPy**, and **Matplotlib**.

## 📌 What is Multimodal Deep Learning?
Multimodal AI models combine data from multiple distinct modalities—such as **Visual Features (Images)** and **Text Embeddings (Language)**—into a unified representation space to perform complex classification, captioning, or visual question answering (VQA) tasks.

---

## 🛠️ Step 1: Import Libraries & Generate Multimodal Synthetic Dataset
We create a synthetic dataset consisting of paired **Image Feature Vectors** (from a simulated CNN/Vision Transformer) and **Text Embeddings** (from a simulated BERT/Language Transformer).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Dataset Generation (Image vector dim = 64, Text embedding dim = 128)
class MultimodalSyntheticDataset(Dataset):
    def __init__(self, num_samples=500, img_dim=64, text_dim=128):
        self.img_data = torch.randn(num_samples, img_dim)
        self.text_data = torch.randn(num_samples, text_dim)
        
        # Synthetic Multimodal Ground Truth Target (Binary Classification)
        # Target depends on non-linear interaction of vision + text features
        interaction = (self.img_data[:, :5].sum(dim=1) + self.text_data[:, :5].sum(dim=1))
        self.labels = (interaction > 0).long()
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return self.img_data[idx], self.text_data[idx], self.labels[idx]

# Instantiate DataLoader
dataset = MultimodalSyntheticDataset()
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset created with {len(dataset)} multimodal samples.")
img_sample, text_sample, label_sample = dataset[0]
print(f"Image Feature Shape: {img_sample.shape}, Text Embedding Shape: {text_sample.shape}")

--- 
## 🔹 Step 2: Build Dual-Stream Multimodal Fusion Network Architecture

We construct a PyTorch module with:
1. **Vision Sub-Network:** Processes image spatial vectors.
2. **Text Sub-Network:** Processes textual embedding vectors.
3. **Fusion Layer:** Concatenates both modality embeddings and passes through a Joint Multimodal Classifier.

In [ ]:
class MultimodalFusionNet(nn.Module):
    def __init__(self, img_dim=64, text_dim=128, hidden_dim=64, num_classes=2):
        super(MultimodalFusionNet, self).__init__()
        
        # Modality Encoder 1: Image Branch
        self.img_encoder = nn.Sequential(
            nn.Linear(img_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Modality Encoder 2: Text Branch
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Joint Multimodal Fusion Classifier (Late Fusion)
        self.fusion_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, img_x, text_x):
        # Extract modality-specific representations
        img_feat = self.img_encoder(img_x)
        text_feat = self.text_encoder(text_x)
        
        # Multimodal Concatenation Fusion
        fused_features = torch.cat((img_feat, text_feat), dim=1)
        
        # Classification output
        output = self.fusion_classifier(fused_features)
        return output

model = MultimodalFusionNet()
print(model)

--- 
## 🔹 Step 3: Train the Multimodal Neural Network

We use Cross-Entropy Loss and Adam Optimizer to train the joint fusion model over 20 epochs.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
loss_history = []
acc_history = []

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for imgs, texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(imgs, texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    loss_history.append(epoch_loss)
    acc_history.append(epoch_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%")

--- 
## 🔹 Step 4: Visualize Training Curves & Performance

Plot Loss convergence and Accuracy metrics during multimodal joint training.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(range(1, epochs+1), loss_history, 'r-o', linewidth=2)
ax1.set_title("Multimodal Training Loss Convergence")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True)

# Accuracy plot
ax2.plot(range(1, epochs+1), [a * 100 for a in acc_history], 'g-s', linewidth=2)
ax2.set_title("Multimodal Classification Accuracy (%)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True)

plt.tight_layout()
plt.show()

--- 
## 🎯 Summary

In this tutorial, we demonstrated:
1. **Synthetic Multimodal Data Pipeline:** Combining vision vectors and text embeddings.
2. **Dual-Stream Feature Encoding:** Parallel processing of separate data streams.
3. **Late Fusion Architecture:** Merging cross-modal features into a joint classification layer in PyTorch.
4. **Training & Metrics Visualization.**

Happy Multimodal Deep Learning Coding! 🧠🚀